# Recruitment model-building: driving the faithful mechanism ladder to a locked contract

_Investigation `recruitment-model-building` — coder reproduction notebook._

**Question.** Given the faithful chemotaxis mechanism ladder (static_lambda ->
hill_occupancy -> adaptive_receptor), can a model-building loop be driven
to reproduce, and be graded against, the locked behavior-test contract
that defines "the adaptive model works": recruitment across a range of
background cue levels via fold-change detection, with a fixed-kd
mechanism as the discriminating knockout it must climb past?

Phase 2 home for the recruitment-adaptive locked contract; the full investigation narrative (loop trajectory, verdict) lands in Phase 3 once the loop driver (Task 7) has been run against it.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-cpm/viva-cpm').is_dir():
    REPO = Path('/home/runner/work/viva-cpm/viva-cpm')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from pbg_cpm_studies.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Adaptive receptor recruitment: fold-change detection across background cue levels (`recruitment-adaptive`)

**Question.** Across a RANGE of background cue levels -- not just one -- can responders
keep being recruited to a rising source cue? Fold-change detection
(fold_change_detection) predicts that an adaptive receptor mechanism, which
re-centers its half-occupancy point on the locally sensed mean rather than
fixing it, should recruit at both low and high static backgrounds, while a
fixed-kd (hill_occupancy) mechanism saturates and collapses once the
background rises far enough to pin occupancy near its ceiling on both sides
of the gradient. This study is the LOCKED CONTRACT: it does not itself run
the model -- it defines the behavior-test bands the model-building loop
(Task 7) must clear for "the adaptive model works" to be true.

**Objective.** Author the acceptance contract for the adaptive-receptor rung of the
faithful chemotaxis mechanism ladder (pbg_cpm_studies.model_building.
mechanisms): three primary behavior tests plus a receptor-gating control,
with pass/fail bands set from real measured recruitment_index values
(3-12 seeds) already gathered while calibrating the ladder. The loop
driver built in the next task runs mechanisms.simulate_condition against
these bands; this study does not itself constitute a completed run.

**Hypothesis.** An adaptive receptor mechanism (a slowly-tracked local-mean set-point that
re-centers the Hill kd every SAMPLE-step increment) recruits responders at
low background AND at high background, because it senses the CHANGE in
local occupancy rather than its absolute level (Barkai & Leibler 1997;
Tu, Shimizu & Berg 2008). A fixed-kd hill mechanism recruits at low
background but collapses at high background once both sides of the
gradient saturate near occupancy ceiling. Both mechanisms are abolished
when the downstream chemotactic response is blocked, since neither is a
metric artifact -- only a mechanism that isn't receptor-mediated at all
(the static-lambda control) would ignore that block.

**Purpose.** adaptation

**Claim.** A receptor mechanism that adapts its half-occupancy set-point to the
locally sensed background (fold-change detection) recruits responders
across a RANGE of background cue levels where a fixed-kd mechanism
saturates and fails: measured recruitment_index at low_bg is 0.82
(adaptive) vs 0.58 (hill); at high_bg it is ~0.56 (adaptive, mean over 12
seeds) vs 0.00 (hill, deterministic collapse). Blocking the receptor-gated
response abolishes recruitment for both (0.00), while the non-receptor
static mechanism ignores the block (0.72) -- the discriminating control
that isolates receptor-mediated gating as the necessary cause.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `adaptive` | `pbg_cpm_studies.composites.chemotaxis.recruitment` | 0 | cue_rate=10.0, chemo_lambda=14.0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.chemotaxis.recruitment`** — `spec_pbg_cpm_studies_composites_chemotaxis_recruitment` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.chemotaxis.recruitment` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: recruitment-adaptive ===
STUDY = 'recruitment-adaptive'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| recruits_low | kind=recruitment_index condition=low_bg stat=final | op in_range low 0.4 high 1.0 |
| receptor_gating | kind=recruitment_index condition=high_bg_blocked stat=final | op <= value 0.15 |
| recruits_high | kind=recruitment_index condition=high_bg stat=final | op in_range low 0.35 high 1.0 |
